<a href="https://colab.research.google.com/github/Fizzah-Amir14/flyrank-ml-internship/blob/main/Copy_of_capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fizzah-Amir14/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

In [1]:
import os
# Look inside the repository folders to find the data
print(os.listdir('.'))
if os.path.exists('work'):
    print("Inside work:", os.listdir('work'))

['.config', 'sample_data']


In [2]:
!git clone https://github.com/Fizzah-Amir14/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 154, done.
remote: Counting objects: 100% (154/154), done.
remote: Compressing objects: 100% (106/106), done.
remote: Total 154 (delta 60), reused 104 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (154/154), 1.87 MiB | 9.60 MiB/s, done.
Resolving deltas: 100% (60/60), done.


In [3]:
import os, getpass

# CI and power users set HF_TOKEN in the environment; everyone else gets the safe prompt.
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


Paste your Hugging Face READ token (hf_...): ··········


In [4]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [5]:
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())
clients.head(10)


clients with 12+ months of GSC history: 4


,client_hash_id,access_profile,gsc_data_start,ga4_data_start
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT
5,client_b10cb2997d0c7c86,gsc_and_ga4,2025-06-18,2025-11-15
6,client_65de48885f4ef01b,gsc_and_ga4,2025-06-21,2026-02-19
7,client_c182d11e4862a37d,gsc_and_ga4,2025-06-21,2026-02-20
8,client_3197e6291363b4db,gsc_and_ga4,2025-06-29,2025-11-09
9,client_625b6439094e23e4,gsc_and_ga4,2025-07-01,2026-02-19


In [6]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily_sample']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily_sample']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 0
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

409,205 content items with enough history


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30
0,client_f623b01661d4bfe4,content_08bfdd6c0a86993f,116.0,0.0,1.0,9.4255
1,client_f623b01661d4bfe4,content_4819593b7027a1c2,0.0,0.0,0.0,NaN
2,client_f623b01661d4bfe4,content_efcc3bbd717147ba,0.0,0.0,0.0,NaN
3,client_f623b01661d4bfe4,content_5e3ad564eda5b38a,2.0,0.0,0.0,97.5000
4,client_f623b01661d4bfe4,content_dcb2c5c2d3a50119,0.0,0.0,0.0,NaN


In [7]:
# 1. Generate the qsignals table
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

# 2. Merge features with qsignals to create 'data'
data = features.merge(qsignals, on='content_hash_id', how='left')

# 3. Handle missing values from the merge
# Fill missing query data with 0
zero_cols = ['visible_queries', 'rare_share', 'anon_share', 'top_query_impressions', 'kept_impressions']
data[zero_cols] = data[zero_cols].fillna(0)
data['top_query_share'] = data['top_query_impressions'] / data['kept_impressions'].replace(0, 1)

# 4. Perform your CTR and Position calculations
data['ctr_last30'] = data['clk_last30'] / data['imp_last30'].replace(0, 1)
data['ctr_last30'] = data['ctr_last30'].fillna(0)
data['pos_last30'] = data['pos_last30'].fillna(100.0)

print(f'Final dataset size: {len(data):,} rows')
data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Final dataset size: 409,205 rows


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share,ctr_last30
0,client_f623b01661d4bfe4,content_08bfdd6c0a86993f,116.0,0.0,1.0,9.4255,2.0,0.092593,0.802469,18.0,34.0,0.529412,0.008621
1,client_f623b01661d4bfe4,content_4819593b7027a1c2,0.0,0.0,0.0,100.0000,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000
2,client_f623b01661d4bfe4,content_efcc3bbd717147ba,0.0,0.0,0.0,100.0000,1.0,0.055556,0.055556,16.0,16.0,1.000000,0.000000
3,client_f623b01661d4bfe4,content_5e3ad564eda5b38a,2.0,0.0,0.0,97.5000,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000
4,client_f623b01661d4bfe4,content_dcb2c5c2d3a50119,0.0,0.0,0.0,100.0000,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000


In [8]:
data['ctr_last30'] = data['clk_last30'] / data['imp_last30']
data['ctr_last30'] = data['ctr_last30'].fillna(0) # Handle pages with 0 impressions

In [9]:
#average
data['pos_last30'] = data['pos_last30'].fillna(100.0) # Penalty for no impressions


In [10]:
data['label'] = (data['imp_last30'] < data['imp_prev30'] * 0.99).astype(int)


In [11]:
from sklearn.model_selection import train_test_split

# 1. Choose your features (X) and your target label (y)
feature_cols = ['ctr_last30', 'pos_last30', 'visible_queries', 'top_query_share', 'rare_share', 'anon_share']
X = data[feature_cols]
y = data['label']

# 2. Split the data into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Print the shapes to confirm it worked
print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")

Training features shape: (327364, 6)
Testing features shape: (81841, 6)


In [13]:
print(data['label'].value_counts())

label
0    409205
Name: count, dtype: int64


In [14]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# 1. Set a new binary label based on the median position (81.7)
# 1 = Worse ranking than average, 0 = Better ranking than average
data['label'] = (data['pos_last30'] > 81.7).astype(int)

print("New Label Distribution:")
print(data['label'].value_counts())
print("-" * 40)

# 2. Select input features (leaving out pos_last30 so the model doesn't cheat)
feature_cols = ['ctr_last30', 'visible_queries', 'top_query_share', 'rare_share', 'anon_share']
X = data[feature_cols]
y = data['label']

# 3. Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Train the model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

print("\nModel training complete!")

New Label Distribution:
label
1    204603
0    204602
Name: count, dtype: int64
----------------------------------------

Model training complete!


In [15]:
# Check the math behind the impressions columns
print("Average of imp_last30:", data['imp_last30'].mean())
print("Average of imp_prev30:", data['imp_prev30'].mean())

# Find the median of our position feature to use as a backup label
median_pos = data['pos_last30'].median()
print("Median of pos_last30:", median_pos)

Average of imp_last30: 528.3290086875771
Average of imp_prev30: 0.0
Median of pos_last30: 81.70382543382543


In [16]:
from sklearn.metrics import classification_report, accuracy_score

# 1. Have the model guess 0 or 1 for the test data using its learned weights
y_pred = model.predict(X_test)

# 2. Check the overall accuracy score
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.4f}\n")

# 3. Print the detailed breakdown report (Precision and Recall)
print("Detailed Performance Report:")
print(classification_report(y_test, y_pred))

Model Accuracy: 0.8211

Detailed Performance Report:
              precision    recall  f1-score   support

           0       0.96      0.67      0.79     40830
           1       0.75      0.97      0.85     41011

    accuracy                           0.82     81841
   macro avg       0.85      0.82      0.82     81841
weighted avg       0.85      0.82      0.82     81841



## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.